# Conditional Edge

## 조건부 Edge(`Conditional Edge`) 사용해 분기 만들기
* `Conditional Edge`를 이용해 상황에 따라 다른 노드로 연결되도록 구현할 수 있습니다.

In [ ]:
from typing import Literal, Annotated
from pydantic import BaseModel, Field
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage, HumanMessage

class AgentState(BaseModel):
    # Annotated[타입, 메타데이터] -> 해당 변수에 '메타데이터'에 해당하는 특성(기능)을 부여함
    # add_messages -> message에 적용하는 메타데이터, Node에서 history에 추가할 항목만 반환해도 알아서 리스트에 추가해 줌
    chat_history: Annotated[list[BaseMessage], add_messages] = Field(default_factory=list) # 맨 처음 생성 시 빈 리스트로 시작
    last_player: Literal['A', 'B']
    number: int
    end: bool

In [39]:
import time
def node_A(state: AgentState) -> dict:
    num = state.number
    new_num = num * 2
    time.sleep(2)
    return {
        'chat_history': [HumanMessage(name='A', content=f"숫자를 {num}에서 {new_num}(으)로 바꿈")],
        'number': new_num,
        'last_player': 'A'
    }

In [40]:
def node_B(state: AgentState) -> dict:
    num = state.number
    new_num = num +20
    time.sleep(2)
    return {
        'chat_history': [HumanMessage(name='B', content=f"숫자를 {num}에서 {new_num}(으)로 변경")],
        'number': new_num,
        'last_player': 'B'
    }

In [41]:
def node_judge(state: AgentState) -> dict:
    end = True if state.number > 100 else False
    return {
        'chat_history': [HumanMessage(name='judge', content=f"최종 숫자 {state.number}(으)로 종료합니다" if end else "계속 진행하세요")],
        'end': end
    }

In [42]:
from langgraph.graph import START, END, StateGraph

workflow = StateGraph(AgentState)

workflow.add_node('A', node_A)
workflow.add_node('B', node_B)
workflow.add_node('judge', node_judge)

workflow.add_edge(START, 'A')
workflow.add_edge('A', 'judge')
workflow.add_edge('B', 'judge')

## `route` 만들고 `conditional edge`에 부여하기
* `route`는 조건에 따라서 달라지는 목적지 `node`명 을 반환하는 함수입니다
* `workflow.add_conditional_edges`함수에 `route`를 적용하면, route가 반환한 `node`명으로 라우팅됩니다.

In [43]:
def judge_route(state: AgentState):
    # end가 True면 END로,
    # 짝수면 A로, 홀수면 B로 가는 route
    if state.end:
        return END
    elif state.last_player == 'B':
        return 'A'
    else:
        return 'B'

workflow.add_conditional_edges(
    'judge', 
    judge_route
)

In [44]:
app = workflow.compile()

init_input = AgentState(chat_history=[], last_player='A', number=3, end=False)

for chunk in app.stream(init_input):
    print(chunk)

{'A': {'chat_history': [HumanMessage(content='숫자를 3에서 6(으)로 바꿈', additional_kwargs={}, response_metadata={}, name='A', id='d0d158f2-c8cb-4b34-9c83-a2242e0256d6')], 'number': 6, 'last_player': 'A'}}
{'judge': {'chat_history': [HumanMessage(content='계속 진행하세요', additional_kwargs={}, response_metadata={}, name='judge', id='cc55ac6e-3a40-4f70-9ed0-982e8557f1a0')], 'end': False}}
{'B': {'chat_history': [HumanMessage(content='숫자를 6에서 26(으)로 변경', additional_kwargs={}, response_metadata={}, name='B', id='c54bc090-e080-47ff-9a91-c8acee2ba9f3')], 'number': 26, 'last_player': 'B'}}
{'judge': {'chat_history': [HumanMessage(content='계속 진행하세요', additional_kwargs={}, response_metadata={}, name='judge', id='1d341cfc-2a2d-4dae-bb6c-7100eae1bece')], 'end': False}}
{'A': {'chat_history': [HumanMessage(content='숫자를 26에서 52(으)로 바꿈', additional_kwargs={}, response_metadata={}, name='A', id='cd8575c2-ff86-4d04-9c1e-ae1b438015cc')], 'number': 52, 'last_player': 'A'}}
{'judge': {'chat_history': [HumanMessage(co

## 토마토 게임 만들기
* ai와 토마토 게임을 진행할 수 있는 LangGraph 기반 게임을 만들어 봅시다
* 규칙: 두 명이 번갈아가면서 '토마토'를 한글자씩 말합니다. 올바르게 말하지 못하면 패배합니다.

In [1]:
from typing import Literal, Annotated,Optional, Any
from pydantic import BaseModel, Field
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage, HumanMessage

class TomatoGameState(BaseModel):
    next_player: Literal['user', 'ai']
    end : bool
    player_input : str =""
    number : int = 0


In [7]:
import random

def start_node(state: TomatoGameState) -> dict:
    # [추가] 게임 시작 시 50% 확률로 user 또는 ai를 첫 플레이어로 랜덤 선택
    first_player = random.choice(['user', 'ai'])
    print(f"첫 번째 플레이어: {first_player}")
    return {'next_player': first_player}

def user_node(state: TomatoGameState) -> dict:
    user_input = input("당신의 대답: ")
    return {
        'player_input': state.player_input+user_input,
        'next_player' :'ai',
        'number' : state.number +1
    }

def ai_node(state: TomatoGameState) -> dict:
    ai_input = random.choice(['토', '마','토'])  # 50% 확률로 둘 중 하나
    print(f"AI의 대답: {ai_input}")
    return {
        'player_input': state.player_input + ai_input,
        'next_player': 'user',   # AI 차례 끝나면 user 차례
        'number': state.number + 1
    }

def judge_node(state: TomatoGameState) -> dict:
    # 토마토 게임 규칙: '토마토'를 한 글자씩 반복 → 토, 마, 토, 토, 마, 토, 토, 마, 토, ...
    tomato_pattern = ('토', '마', '토')

    for i, char in enumerate(state.player_input):
        expected = tomato_pattern[i % 3]
        if char != expected:
            # next_player는 다음 차례 → 방금 말한 사람은 그 반대
            loser = 'user' if state.next_player == 'ai' else 'ai'
            print(f"틀렸습니다! '{char}' (X) → '{expected}' (O) 이어야 합니다.")
            print(f"{loser} 패배! 게임 종료")
            return {'end': True}

    # 모든 글자가 순서에 맞으면 계속 진행
    return {'end': False}
    

In [8]:
from langgraph.graph import START, END, StateGraph

# [수정] AgentState → TomatoGameState (토마토 게임 전용 State 사용)
workflow = StateGraph(TomatoGameState)

workflow.add_node('start', start_node)  # [추가] 시작 플레이어 랜덤 결정 노드
workflow.add_node('user', user_node)
workflow.add_node('ai', ai_node)
workflow.add_node('judge', judge_node)

# [수정] 기존 A/B 예제 edge 제거, 토마토 게임 흐름으로 교체
workflow.add_edge(START, 'start')  # START → start 노드
workflow.add_edge('user', 'judge')   # user 말한 뒤 → judge
workflow.add_edge('ai', 'judge')     # ai 말한 뒤 → judge

In [9]:
def start_route(state: TomatoGameState):
    # [추가] start_node가 정한 next_player 값에 따라 user 또는 ai로 분기
    return state.next_player

def judge_route(state: TomatoGameState):
    # [수정] AgentState → TomatoGameState
    # end가 True면 END로, 아니면 next_player에 따라 user/ai로 이동
    if state.end:
        return END
    elif state.next_player == 'user':
        return 'user'
    else:
        return 'ai'

# [추가] START 직후 랜덤으로 선택된 플레이어 노드로 이동
workflow.add_conditional_edges('start', start_route)

workflow.add_conditional_edges('judge', judge_route)

app = workflow.compile()

# [수정] 초기값은 빈 상태로 시작 (첫 플레이어는 start_node가 랜덤 결정)
init_input = TomatoGameState(
    next_player='user',  # start_node에서 덮어씀
    end=False,
    player_input="",
    number=0
)

for chunk in app.stream(init_input):
    print(chunk)


첫 번째 플레이어: ai
{'start': {'next_player': 'ai'}}
AI의 대답: 토
{'ai': {'player_input': '토', 'next_player': 'user', 'number': 1}}
{'judge': {'end': False}}


{'user': {'player_input': '토마', 'next_player': 'ai', 'number': 2}}
{'judge': {'end': False}}
AI의 대답: 토
{'ai': {'player_input': '토마토', 'next_player': 'user', 'number': 3}}
{'judge': {'end': False}}
{'user': {'player_input': '토마토토', 'next_player': 'ai', 'number': 4}}
{'judge': {'end': False}}
AI의 대답: 토
{'ai': {'player_input': '토마토토토', 'next_player': 'user', 'number': 5}}
틀렸습니다! '토' (X) → '마' (O) 이어야 합니다.
ai 패배! 게임 종료
{'judge': {'end': True}}


### 팬아웃(Fan-Out) 구현하기
* 하나의 노드에서 여러 노드로 갈라지고(팬아웃) 다시 한 노드로 흐름을 모을 (팬인) 수 있습니다

In [11]:
from pydantic import BaseModel
from typing import Optional, Any

class FanOutState(BaseModel):
    value1: Optional[Any] = None
    value2: Optional[Any] = None

In [15]:
import time

def node_A(state: FanOutState) -> dict:
    print("A 노드에서 출발합니다.")
    return {"from_node": "A"}

def node_B(state: FanOutState) -> dict:
    for i in range(1, 4):
        print(f"B 노드 실행 중... ({i}/3)")
        time.sleep(1)
    return {"value1": state.value1 + 2}

def node_C(state: FanOutState) -> dict:
    for i in range(1, 4):
        print(f"C 노드 실행 중... ({i}/3)")
        time.sleep(1)
    return {"value2": state.value2 + 1}

In [16]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(FanOutState)

workflow.add_node('A', node_A)
workflow.add_node('B', node_B)
workflow.add_node('C', node_C)

workflow.add_edge(START, 'A')
workflow.add_edge('A', 'B')
workflow.add_edge('A', 'C')

app = workflow.compile()

In [17]:
init_state = FanOutState(value1=0, value2=0)
result = app.invoke(init_state)
result

A 노드에서 출발합니다.
B 노드 실행 중... (1/3)
C 노드 실행 중... (1/3)
B 노드 실행 중... (2/3)C 노드 실행 중... (2/3)

C 노드 실행 중... (3/3)B 노드 실행 중... (3/3)



{'value1': 2, 'value2': 1}

## 팬인(Fan-In) 구현하기
* `add_edge` 함수의 출발점은 여러 노드의 배열도 입력받을 수 있습니다.

In [18]:
def node_D(state: FanOutState):
    for i in range(1, 4):
        print(f"D 노드 실행 중... ({i}/3)")
        time.sleep(1)
    return {"value2": state.value1 + 1}

In [19]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(FanOutState)

workflow.add_node('A', node_A)
workflow.add_node('B', node_B)
workflow.add_node('C', node_C)
workflow.add_node('D', node_D)

workflow.add_edge(START, 'A')
workflow.add_edge('A', 'B')
workflow.add_edge('A', 'C')
workflow.add_edge(['B', 'C'], 'D')
workflow.add_edge('D', END)

app = workflow.compile()

In [20]:
init_state = FanOutState(value1=0, value2=0)
result = app.invoke(init_state)
result

A 노드에서 출발합니다.
B 노드 실행 중... (1/3)
C 노드 실행 중... (1/3)
B 노드 실행 중... (2/3)C 노드 실행 중... (2/3)

C 노드 실행 중... (3/3)B 노드 실행 중... (3/3)

D 노드 실행 중... (1/3)
D 노드 실행 중... (2/3)
D 노드 실행 중... (3/3)


{'value1': 2, 'value2': 3}

### Q. 팬인 시 노드 종료 시점이 다르다면?

In [27]:
import time

def node_A(state: FanOutState) -> dict:
    print("A 노드에서 출발합니다.")
    return {"from_node": "A"}

def node_B(state: FanOutState) -> dict: # 두 노드의 실행 시간을 다르게 하고 결과를 확인해보세요.
    for i in range(1, 10):
        print(f"B 노드 실행 중... ({i}/9)")
        time.sleep(1)
    return {"value1": state.value1 + 1}


def node_C(state: FanOutState) -> dict:
    for i in range(1, 4):
        print(f"C 노드 실행 중... ({i}/3)")
        time.sleep(1)
    return {"value2": state.value2 + 2}

In [28]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(FanOutState)

workflow.add_node('A', node_A)
workflow.add_node('B', node_B)
workflow.add_node('C', node_C)
workflow.add_node('D', node_D)

workflow.add_edge(START, 'A')
workflow.add_edge('A', 'B')
workflow.add_edge('A', 'C')
workflow.add_edge(['B', 'C'], 'D')
workflow.add_edge('D', END)

app = workflow.compile()

In [29]:
init_state = FanOutState(value1=0, value2=0)
result = app.invoke(init_state)
result

A 노드에서 출발합니다.
B 노드 실행 중... (1/9)
C 노드 실행 중... (1/3)
B 노드 실행 중... (2/9)
C 노드 실행 중... (2/3)
C 노드 실행 중... (3/3)
B 노드 실행 중... (3/9)
B 노드 실행 중... (4/9)
B 노드 실행 중... (5/9)
B 노드 실행 중... (6/9)
B 노드 실행 중... (7/9)
B 노드 실행 중... (8/9)
B 노드 실행 중... (9/9)
D 노드 실행 중... (1/3)
D 노드 실행 중... (2/3)
D 노드 실행 중... (3/3)


{'value1': 1, 'value2': 2}

# `Reducer` 활용하기

In [30]:
import time

def node_A(state: FanOutState) -> dict:
    print("A 노드에서 출발합니다.")
    return {"from_node": "A"}

def node_B(state: FanOutState) -> dict:
    for i in range(1, 6):
        print(f"B 노드 실행 중... ({i}/5)")
        time.sleep(1)
    return {"value1": state.value1 + 1}

def node_C(state: FanOutState) -> dict: # B, C 노드 모두에서 value1 값을 변경하는 경우
    for i in range(1, 4):
        print(f"C 노드 실행 중... ({i}/3)")
        time.sleep(1)
    return {"value1": state.value1 + 1}

    

In [31]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(FanOutState)

workflow.add_node('A', node_A)
workflow.add_node('B', node_B)
workflow.add_node('C', node_C)
workflow.add_node('D', node_D)

workflow.add_edge(START, 'A')
workflow.add_edge('A', 'B')
workflow.add_edge('A', 'C')
workflow.add_edge(['B', 'C'], 'D')
workflow.add_edge('D', END)

app = workflow.compile()

In [32]:
init_state = FanOutState(value1=0, value2=0)
result = app.invoke(init_state) # reducer가 없으면 오류 발생
result

A 노드에서 출발합니다.
B 노드 실행 중... (1/5)
C 노드 실행 중... (1/3)
B 노드 실행 중... (2/5)C 노드 실행 중... (2/3)

C 노드 실행 중... (3/3)B 노드 실행 중... (3/5)

B 노드 실행 중... (4/5)
B 노드 실행 중... (5/5)


InvalidUpdateError: At key 'value1': Can receive only one value per step. Use an Annotated key to handle multiple values.
For troubleshooting, visit: https://docs.langchain.com/oss/python/langgraph/errors/INVALID_CONCURRENT_GRAPH_UPDATE

In [33]:
from typing import Annotated

def numberAddReducer(left: int, right: int) -> int: # left: 기존 State, right: return 받은 state
    return left + right # return 받은 값으로 대체하지 않고, 기존 값에 return 받은 값을 더함

class ReducerState(BaseModel):
    value1: Annotated[Optional[Any], numberAddReducer] = None
    value2: Optional[Any] = None

In [46]:
def node_B(state: ReducerState) -> dict: # state 정의도 모두 변경
    for i in range(1, 6):
        print(f"B 노드 실행 중...({i}/5)")
        time.sleep(1)
    return {"value1": 1} # 새로 더해줄 값만 반환하면 됨

def node_C(state: ReducerState) -> dict: # B, C 노드 모두에서 value1 값을 변경하는 경우
    for i in range(1, 4):
        print(f"C 노드 실행 중..({i}/3)")
        time.sleep(1)
    return {"value1": 1}

def node_D(state: ReducerState):
    for i in range(1, 4):
        print(f"D 노드 실행 중...({i}/3)")
        time.sleep(1)
    return {"value2": state.value2+1}


In [47]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(ReducerState)

workflow.add_node('A', node_A)
workflow.add_node('B', node_B)
workflow.add_node('C', node_C)
workflow.add_node('D', node_D)

workflow.add_edge(START, 'A')
workflow.add_edge('A', 'B')
workflow.add_edge('A', 'C')
workflow.add_edge(['B', 'C'], 'D')
workflow.add_edge('D', END)

app = workflow.compile()

In [48]:
init_state = ReducerState(value1=0, value2=0)
result = app.invoke(init_state) # reducer가 없으면 오류 발생
result

A 노드에서 출발합니다.
B 노드 실행 중...(1/5)
C 노드 실행 중..(1/3)
B 노드 실행 중...(2/5)
C 노드 실행 중..(2/3)
B 노드 실행 중...(3/5)
C 노드 실행 중..(3/3)
B 노드 실행 중...(4/5)
B 노드 실행 중...(5/5)
D 노드 실행 중...(1/3)
D 노드 실행 중...(2/3)
D 노드 실행 중...(3/3)


{'value1': 2, 'value2': 1}

### 동적 팬아웃 구현하기
* `Conditional Edge`를 이용하면 Fan-Out을 동적으로 설정할 수 있습니다
* Fan-Out을 설정하려면, 목적지 노드 이름을 리스트로 모두 전달하면 됩니다.

In [43]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(ReducerState)

workflow.add_node('A', node_A)
workflow.add_node('B', node_B)
workflow.add_node('C', node_C)
workflow.add_node('D', node_D)

workflow.add_edge(START, 'A')

def fanout_router(state: ReducerState):
    if state.value2 > 0:
        return 'B'
    else:
        return ['B', 'C']
    
workflow.add_conditional_edges('A', fanout_router)

workflow.add_edge(['B', 'C'], 'D')

def end_router(state: ReducerState):
    if state.value1 > 2:
        return END
    else:
        return 'A'

workflow.add_conditional_edges('D', end_router)


app = workflow.compile()

In [44]:
init_state = ReducerState(value1=0, value2=0)
result = app.invoke(init_state)
result

A 노드에서 출발합니다.
B 노드 실행 중...(1/5)
C 노드 실행 중..(1/3)
C 노드 실행 중..(2/3)B 노드 실행 중...(2/5)

C 노드 실행 중..(3/3)B 노드 실행 중...(3/5)

B 노드 실행 중...(4/5)
B 노드 실행 중...(5/5)
D 노드 실행 중...(1/3)
D 노드 실행 중...(2/3)
D 노드 실행 중...(3/3)
A 노드에서 출발합니다.
B 노드 실행 중...(1/5)
B 노드 실행 중...(2/5)
B 노드 실행 중...(3/5)
B 노드 실행 중...(4/5)
B 노드 실행 중...(5/5)


{'value1': 3, 'value2': 1}

### 라우팅 심화 문제
* Fan-Out과 Reducing, 조건부 Edge를 적극적으로 활용해 음식점 운영을 최적화해봅시다 (괄호 안의 숫자는 소요시간(초))
```
burger: patty(5), bun(3) 각각 필요 -> burgerSet(2)
fries: fry(3) -> friesSet(1)
beverage: beverage(2)

세 가지가 모두 준비되면 종료
```

In [2]:
from pydantic import BaseModel
from typing import List, Annotated, Literal, Optional

def menuReducer(left:List[str], right:List[str]) -> List[str]:
    return left + right

class RestaurantState(BaseModel):
    menu_ordered: List[Literal['burger', 'fries', 'beverage']]
    menu_complete: Annotated[Optional[List[Literal['burger', 'fries', 'beverage']]], menuReducer] = []


In [3]:
import time
from threading import Lock

_print_lock = Lock()

def progress(label: str, seconds: int) -> None:
    """초 단위 작업을 1초마다 진행률과 함께 출력"""
    for i in range(1, seconds + 1):
        pct = int(i / seconds * 100)
        with _print_lock:
            print(f"{label} 진행 중... ({i}/{seconds}) {pct}%", flush=True)
        time.sleep(1)

def log(message: str) -> None:
    with _print_lock:
        print(message, flush=True)


# 각종 그래프 노드 실행 및 상태 갱신 통합: 반환값으로 state의 menu_complete를 적절히 업데이트하도록 구성
def g_burger_start(state: RestaurantState) -> dict:
    log("[버거] 준비 시작")
    return {}

def g_fries_start(state: RestaurantState) -> dict:
    log("[감자튀김] 준비 시작")
    return {}

def g_patty(state: RestaurantState) -> dict:
    progress("[버거] 패티 조리", 5)
    return {}

def g_bun(state: RestaurantState) -> dict:
    progress("[버거] 번 조리", 3)
    return {}

def g_burger_set(state: RestaurantState) -> dict:
    progress("[버거] 세트 조립", 2)
    log("[버거] 완성 ✓")
    return {'menu_complete': ['burger']}

def g_fry(state: RestaurantState) -> dict:
    progress("[감자튀김] 튀김 조리", 3)
    return {}

def g_fries_set(state: RestaurantState) -> dict:
    progress("[감자튀김] 세트 포장", 1)
    log("[감자튀김] 완성 ✓")
    return {'menu_complete': ['fries']}

def g_beverage(state: RestaurantState) -> dict:
    progress("[음료] 준비", 2)
    return {'menu_complete': ['beverage']}


In [7]:
from langgraph.graph import StateGraph, START, END

def g_judge(state: RestaurantState) -> dict:
    ordered = state.menu_ordered
    completed = state.menu_complete or []
    pct = int(len(completed) / len(ordered) * 100) if ordered else 100
    log(f"\n{'='*45}")
    log(f"전체 준비: {len(completed)}/{len(ordered)} ({pct}%)")
    log(f"완료 메뉴: {completed}")
    log(f"{'='*45}")
    return {}

workflow = StateGraph(RestaurantState)

workflow.add_node('g_burger_start', g_burger_start)
workflow.add_node('g_fries_start', g_fries_start)
workflow.add_node('g_patty', g_patty)
workflow.add_node('g_bun', g_bun)
workflow.add_node('g_burger_set', g_burger_set)
workflow.add_node('g_fry', g_fry)
workflow.add_node('g_fries_set', g_fries_set)
workflow.add_node('g_beverage', g_beverage)
workflow.add_node('g_judge', g_judge)

def order_fanout(state: RestaurantState):
    # 주문 메뉴에 따라 병렬로 준비 시작 (동적 Fan-Out)
    nodes = []
    if 'burger' in state.menu_ordered:
        nodes.append('g_burger_start')
    if 'fries' in state.menu_ordered:
        nodes.append('g_fries_start')
    if 'beverage' in state.menu_ordered:
        nodes.append('g_beverage')
    return nodes

def burger_fanout(state: RestaurantState):
    # 패티(5초), 번(3초)을 동시에 조리 (Fan-Out)
    return ['g_patty', 'g_bun']

def end_router(state: RestaurantState):
    # 주문한 메뉴가 모두 완료되면 종료
    if set(state.menu_ordered).issubset(set(state.menu_complete or [])):
        return END

# START → 주문 메뉴별 병렬 분기
workflow.add_conditional_edges(START, order_fanout)

# 버거: patty + bun 병렬 → burgerSet(완료)
workflow.add_conditional_edges('g_burger_start', burger_fanout)
workflow.add_edge(['g_patty', 'g_bun'], 'g_burger_set')

# 감자튀김: fry → friesSet(완료)
workflow.add_edge('g_fries_start', 'g_fry')
workflow.add_edge('g_fry', 'g_fries_set')

# 세 메뉴가 모두 완료되면 판정 (Fan-In)
workflow.add_edge(['g_burger_set', 'g_fries_set', 'g_beverage'], END)
#workflow.add_conditional_edges('g_judge', end_router)

app = workflow.compile()


In [8]:
init_state = RestaurantState(menu_ordered=['burger',  'beverage'])
log(f"주문 접수: {init_state.menu_ordered}\n")
result = app.invoke(init_state)
result

주문 접수: ['burger', 'beverage']

[음료] 준비 진행 중... (1/2) 50%
[버거] 준비 시작
[음료] 준비 진행 중... (2/2) 100%
[버거] 번 조리 진행 중... (1/3) 33%
[버거] 패티 조리 진행 중... (1/5) 20%
[버거] 번 조리 진행 중... (2/3) 66%
[버거] 패티 조리 진행 중... (2/5) 40%
[버거] 번 조리 진행 중... (3/3) 100%
[버거] 패티 조리 진행 중... (3/5) 60%
[버거] 패티 조리 진행 중... (4/5) 80%
[버거] 패티 조리 진행 중... (5/5) 100%
[버거] 세트 조립 진행 중... (1/2) 50%
[버거] 세트 조립 진행 중... (2/2) 100%
[버거] 완성 ✓


{'menu_ordered': ['burger', 'beverage'],
 'menu_complete': ['beverage', 'burger']}